## Simple LLM application with LCEL

Here in this quickstart guide, we will see how to use LCEL to create a simple LLM application.

With this you can have an high level overview of:

- Using language models
- Using PromptTemplates and OutputParsers
- Using Langchain Expression Langugae (LCEL) to chain componentes together
- Debugging and tracing your application with Langsmith
- Deploying your application with LangServe

In [2]:
pip show langchain

Name: langchain
Version: 1.2.18
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: e:\TCS\Langchain\venv\Lib\site-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [5]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

In [6]:
from langchain_groq import ChatGroq
llm = ChatGroq(model="llama-3.1-8b-instant", api_key=groq_api_key)
llm

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002552EFCA900>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002552EFCB380>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [21]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

messages = [
    SystemMessage(content="You are en expert transalator who is capable of transalating the sentnece that user gives to french the user can give whatever sentence they want in whichever language they want you need to translate it to French.\n Make sure to only give the transalated sentence as output actual sentence as input and nothing else. Well format the input and output in the following way:\n\nInput: <the sentence that needs to be transalted>\nOutput: <the transalated sentence in french>"),
    HumanMessage(content="Hi friend, How are you!")
]


In [22]:
chain = llm|parser
chain.invoke(messages)

'Input: Hi friend, How are you!\nOutput: Bonjour, comment vas-tu !'

In [23]:
## Instead of giving the messages repeatedly you can use prompt templates

from langchain_core.prompts import ChatPromptTemplate

generic_template = """You are en expert transalator who is capable of transalating the sentnece that user gives to {language} the user can give whatever sentence they want in whichever language they want you need to translate it to {language}.
Make sure to only give the transalated sentence as output actual sentence as input and nothing else"""

prompt = ChatPromptTemplate.from_messages([
    ("system", generic_template),
    ("user", "{text}")
])



In [25]:
result = prompt.invoke({
    "language": "french",
    "text": "Hi friend, How are you!"
})

result

ChatPromptValue(messages=[SystemMessage(content='You are en expert transalator who is capable of transalating the sentnece that user gives to french the user can give whatever sentence they want in whichever language they want you need to translate it to french.\nMake sure to only give the transalated sentence as output actual sentence as input and nothing else', additional_kwargs={}, response_metadata={}), HumanMessage(content='Hi friend, How are you!', additional_kwargs={}, response_metadata={})])

In [26]:
result.to_messages()

[SystemMessage(content='You are en expert transalator who is capable of transalating the sentnece that user gives to french the user can give whatever sentence they want in whichever language they want you need to translate it to french.\nMake sure to only give the transalated sentence as output actual sentence as input and nothing else', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hi friend, How are you!', additional_kwargs={}, response_metadata={})]

In [30]:
chain = prompt|llm|parser
chain.invoke({
    "language": "tamil",
    "text": "Hi friend, How are you! TVK is the best"
})

'வணக்கம் நண்பனே, நீங்கள் எப்படி இருக்கிறீர்கள்! TVK சிறந்தது.'